[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LizbethMG-Teaching/pose2behav-book/blob/main/notebooks/analysis_multi-animal.ipynb)

# 📓 Notebook 3 – Analysis of multi animal (top-view mouse)
## 1. Introduction & objectives

In this notebook, you will analyze pose estimation outputs generated with the SuperAnimal ModelZoo on a top view multi animal video containing several mice.

**Learning goals:**

After this notebook, you should be able to:
- Load and preprocess multi animal pose data from SuperAnimal DLC
- Implement your own filtering and interpolation choices
- Compute activity and social metrics per mouse
- Integrate the results into a summary table

--- 

**About this notebook**

In this notebook, you will analyze pose-estimation data from freely-moving mice. 

# 🐭🐭🏠🎥 The Mouse House: multi animal pose challenge

Five mice live together in the "Mouse House" 🐭🤍🏠, a fully monitored arena.
Every movement is tracked with SuperAnimal DeepLabCut.

Your task is to use pose data to build a behavioral profile for each mouse:
- Who is the Hyperactive One?
- Who is the Social Butterfly?
- Who is the Lone Wolf?
- And who wins each "medal" category?

🥇🥈🥉 At the end, you will assign gold, silver, and bronze medals in:
- Activity
- Sociability

You will work mostly independently, but everyone must use the same output variable names and structure so we can compare results.

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/single-frame-multi.png" width="50%">

--- 
**Instructions**

This notebook mixes pre-filled code cells (ready to run) and coding exercises that you will complete.

- Some cells are already complete (just run them).
- When you see a cell with a TODO, you must write code.
- You are free to choose methods, but you must respect:
  - Input: the provided pose file
  - Output variable names and column names as indicated. 

👉 Here’s how to work through it:
1. Read carefully each section before running the cells.
2. When a cell requires you to code, you’ll see a TODO comment.
3. The TODO will tell you how many lines of code you are expected to write.
4. Write your code only between the markers:
    
```python
# >>>>>>>>>>>>>>>>>>>
# your code goes here
# <<<<<<<<<<<<<<<<<<<
```

✋ Do not edit anything outside these markers.

⚡ After finishing the course, feel free to experiment and modify the notebook as you like!

---




## 2. Data Loading & Format Inspection

### 2.1 Download data (prefilled)

**📋 Instructions:**
- Run the code cell below to download the dataset file.

In [ ]:
# PREFILLED, NO NEED TO CHANGE, JUST RUN THIS CELL
# Install and import the required libraries:
!pip -q install gdown tables

import os
from pathlib import Path
import gdown, pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import groupby
import re
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle


# --------------------------------------------------------------

# Detect if running in Google Colab
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    DEST = Path("/content/cleaned_pose_downloaded.h5")
else:
    DEST = Path("cleaned_pose_downloaded.h5")  # save in current folder locally
print("Saving to:", DEST)

# Select here the experiment you want to download, comment the others:
# Opt 1: Single mouse - arena without clear floor
FILE_ID = "1H-Qf8i-Y7bN4MdUT3yHyi-yVi5Aw7Emd"
# Opt 2: Single mouse - arena with bedding
# FILE_ID = 


URL = f"https://drive.google.com/uc?id={FILE_ID}"

print("Downloading from Drive...")
_ = gdown.download(URL, str(DEST), quiet=False)

# Basic checks
assert DEST.exists() and DEST.stat().st_size > 0, "❌ Download failed or empty file."
print(f"✅ Downloaded to {DEST} ({DEST.stat().st_size/1_000_000:.2f} MB)")

# --- Load the cleaned H5 file into a pandas DataFrame ---
df = pd.read_hdf(DEST, key="df_with_missing")

print("✅ Data loaded successfully!")
print("Shape:", df.shape)
print("Columns:", list(df.columns)[:8], "...")